# Heart Disease Risk Factor Analysis

### Dataset
UCI Heart Disease Dataset (Cleveland) — kaggle.com/datasets/cherngs/heart-disease-cleveland-uci — 303
patients, 14 features, free, no registration required.

### Clinical Context
303 anonymised patient records with 14 clinical attributes: age, sex, chest pain type, resting BP, cholesterol,
fasting glucose, ECG findings, max heart rate, and exercise-induced angina. 

### Key Questions
- Which clinical risk factors are most strongly associated with heart disease?
- Does the pattern differ between male and female patients?
- Can a simple age-stratified risk profile be built from descriptive statistics alone, before any ML?

In [ ]:
import sys
print(sys.executable)
print()

import pandas as pd

## Step 1:
### Load and inspect the dataset

In [ ]:
# Load dataset from folder
df = pd.read_csv("../data/raw/heart_cleveland_upload.csv")
df.info()

In [ ]:
# Prints table with summary statistics of each column: count, mean, 
# std, Range, min, and max.  Each column label is shown. 
df.describe()

### Data Audit Summary - Step 1

- Dataset contains 297 patient records across 14 columns, with no missing values.
- Resting blood pressure (trestbps) ranges from 94–200 mmHg. The minimum is borderline low
  but not clinically urgent; the maximum is consistent with hypertensive presentations
  commonly seen in chest-pain evaluations and would typically prompt BP-lowering treatment.
- Serum cholesterol (chol) ranges from 126–564 mg/dl. The maximum is a plausible but notable
  outlier, consistent with severe hypercholesterolemia (e.g. familial hypercholesterolemia)
  rather than a data entry error.
- No implausible or out-of-range values were identified that would suggest data quality issues.

In [ ]:
# Check data for null values.  This is done column by column. 
df.isnull().sum()

## Step 2

### Clean and Label Dataset
- Impute missing data with its column's median: not necessary here given no null values. 
- Map numeric codes to clinical labels

### Mapping numeric codes to clinical labels

In [ ]:
# Define the mapping as a dictionary which becomes a lookup table, 
# replacing each value in a specified column according to it.
# In this case, chest pain characterization. 

cp_labels = {
    0: "typical angina",
    1: "atypical angina",
    2: "non-anginal pain",
    3: "asymptomatic"
}

# Create a new labeled column using .map()
df['cp_label'] = df['cp'].map(cp_labels)

# Print new column side by side with existing column and verify mapping
# using first 5 rows ".head()". 
df[['cp', 'cp_label']].head()

In [ ]:
# Confirm for other categories using .value_counts()
df['cp_label'].value_counts()

In [ ]:
# Create mapping dictionary for sex categories 

sex_labels = {
    0: "female",
    1: "male"
}

# Create a new labeled column using .map()
df['sex_label'] = df['sex'].map(sex_labels)

# Print new column side by side with existing column and verify mapping
# using first 5 rows ".head()". 
df[['sex', 'sex_label']].head()

In [ ]:
# Confirm for all values using .value_counts()
df['sex_label'].value_counts()

In [ ]:
# Creaste mapping dictionary for "fbs", fasting blood glucose column

fbs_labels = {
    0: "false",
    1: "true"
}

# Create a new labeled column using .map()
df['fbs_label'] = df['fbs'].map(fbs_labels)

# Side by side comparison with existing column using first 5 values .head().
df[['fbs', 'fbs_label']].head()

In [ ]:
# Confirm for all row values using .value_counts().
df['fbs_label'].value_counts()



In [ ]:
# Create mapping dictonary for "restecg", resting ECG reading column
restecg_labels = {
    0: "normal",
    1: "ST-T Wave abnormality",
    2: "probable/definite LVH (Estes' Criteria)" 
}

# Create a new labled column using .map()
df['restecg_label'] = df['restecg'].map(restecg_labels)

# Create side by side comparison with existing column using first 5 values .head()
df[['restecg', 'restecg_label']].head()



In [ ]:
# Confirm mapping for all row values in "restecg_label" column using
# .value_counts().

df['restecg_label'].value_counts()




In [ ]:
# Create mapping dictonary for "exang", exercise-induced angina column
exang_labels = {
    0: "no",
    1: "yes"
}

# Create a new labeled column with mapped values using .map().
df['exang_label'] = df['exang'].map(exang_labels)

# Create side by side comparison of new column with existing column using .head().
df[['exang', 'exang_label']].head()



In [ ]:
# Confirm mapping for all row values in the new column using .value_counts().
df['exang_label'].value_counts()


In [ ]:
# Create mapping dictionary for "slope", slope of the peak exercise
# ST segment column.
slope_labels = {
    0: "upsloping",
    1: "flat",
    2: "downsloping"
}

# Create a new labeled column with mapped values using .map().
df['slope_label'] = df['slope'].map(slope_labels)

# Create side by side comparison of new labeled column with existing column
# using .head().
df[['slope', 'slope_label']].head()


In [ ]:
# Confirm mapping for all row values in the new column using .value_counts().
df['slope_label'].value_counts()


In [ ]:
# Create mapping dictionary for "thal", Thalassemia (blood flow) result
# column.
thal_labels = {
    0: "normal",
    1: "fixed defect",
    2: "reversible defect"
}

# Create a new labeled column with mapped values using .map()
df['thal_label'] = df['thal'].map(thal_labels)

# Create side by side comparison of new labeled column with existing column
# using .head().
df[['thal', 'thal_label']].head()


In [ ]:
# Confirm mapping for all rows in the new labeled column, using .value_counts().
df['thal_label'].value_counts()



In [ ]:
# Create mapping dictionary for "Condition", "Target variable - presence of 
# heart disease" column.
condition_labels = {
    0: "no disease",
    1: "disease present"
}

# Create a new labeled column with mapped values using .map().
df['condition_label'] = df['condition'].map(condition_labels)

# Show side by side comparison of new labeled column with existing column
# using .head().
df[['condition', 'condition_label']].head()



In [ ]:
# Confirm mapping for rows in new column, using .value_counts().
df['condition_label'].value_counts()


In [ ]:
# Show table with new columns
df.info()


## Step 3 Univariate EDA

Plot histograms for age, cholesterol, resting BP, and max HR using
plt.subplots(2,2). Annotate each with clinical reference ranges as red dashed lines.

#### Baseline references
- **Chol**:          240 mg/dl (the "high" threshold)
- **trestbps**:      130 mm Hg
- **age**:           50 years
- **thalach**(Max HR achieved ex stress test):  **220 - age**


In [ ]:
import matplotlib.pyplot as plt

# Create column for  predicted max HR during ex stress test
df['predicted_max_hr'] = 220 - df['age']


In [ ]:
# Importing and printing matplotlib CSS4 colors.
import matplotlib.colors as mcolors
print(list(mcolors.CSS4_COLORS.keys()))

### Creating and showing Univariate distributions via histograms

In [ ]:
# Creating a 2 X 2 historgram grid for the four columns above. 
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Creating histogram for "age", plot figure position, axes[0,0]
axes[0, 0].hist(df['age'], bins=15, color='steelblue', edgecolor='black')
axes[0, 0].axvline(50, color='red', linestyle='--', 
                   label='Elevated risk threshold (50)')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age (years)')
axes[0, 0].set_ylabel('Number of Patients')
axes[0, 0].legend()

# Creating histogram for "chol", Cholesterol, plot figure position, axes [0,1]
axes[0,1].hist(df['chol'], bins=15, color='steelblue', edgecolor='black')
axes[0,1].axvline(240, color='red', linestyle='--', 
                  label='High cholesterol (240 mg/dl)')
axes[0, 1].set_title('Cholesterol Distribution')
axes[0, 1].set_xlabel('Serum Cholesterol (mg/dl)')
axes[0, 1].set_ylabel('Number of Patients')
axes[0, 1].legend()

# Creating histogram for "trestbps", Resting Blood Pressure, axes[1,0].
axes[1, 0].hist(df['trestbps'], bins=15, color='steelblue', edgecolor='black')
axes[1, 0].axvline(130, color='red', linestyle='--', label='Stage 1 hypertension (130 mmHg)')
axes[1, 0].set_title('Resting Blood Pressure Distribution')
axes[1, 0].set_xlabel('Resting BP (mm Hg)')
axes[1, 0].set_ylabel('Number of Patients')
axes[1, 0].legend()

# Creating histogram for "thalach", Max HR Achieved, position axes[1,1]
axes[1, 1].hist(df['thalach'], bins=15, color='steelblue', edgecolor='black')
axes[1, 1].set_title('Max Heart Rate Achieved Distribution')
axes[1, 1].set_xlabel('Max Heart Rate (bpm)')
axes[1, 1].set_ylabel('Number of Patients')
# No fixed reference line — predicted max HR is age-dependent,
# addressed in Step 4


# Save and Show plots
plt.tight_layout()

# Saving plot figure to figures subdirectory
fig.savefig('../figures/01_univariate_histograms.png',
             dpi=300, bbox_inches='tight')

# Showing plot in output
plt.show()


### Explanation of the above function call
- **plt** — this is matplotlib.pyplot, the plotting library, imported earlier as plt (a standard convention, like pd for pandas)

- **.subplots(...)** — a function that creates a grid of charts all at once, rather than one chart at a time

- **2, 2** — the grid dimensions: 2 rows, 2 columns — so 4 chart "slots" total, arranged like a 2×2 spreadsheet

- **figsize=(12, 8)** — sets the overall canvas size in inches: 12 wide, 8 tall. This is the size of the whole 2×2 grid together, not each individual chart

- **fig, axes = ...** - returns two separate objects, and Python lets you capture both in one line using a comma. 

- **fig** (short for "figure") — represents the entire canvas/window, the whole 2×2 grid as one unit. **Fig** is used to control something about the whole picture at once — like **fig.suptitle("My Overall Title")** for one title spanning all four charts, or **fig.savefig("figures/histograms.png")** to save the entire grid as one image file.

- **axes** — An array that represents the individual chart slots inside that grid. Since the statement asked for a 2×2 grid, axes isn't just one chart — it's a 2×2 array of chart objects, one for each position. This is why later code says **axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]** — it is reaching into that array to grab one specific chart slot and draw into it individually (row index, column index — same logic as **df.iloc[row, col]** as seen in a pandas pattern, or a spreadsheet cell reference).

- **plt.tight_layout()** just prevents the four charts from overlapping/crowding each other visually.

### Univariate EDA Summary — Step 3

- **Age**: Distribution is roughly bell-shaped, centered in the mid-50s to mid-60s. The majority of the cohort is already past the age-50 elevated-risk threshold, consistent with expected demographics for a cardiac-focused clinical dataset.

- **Cholesterol**: Most patients fall below the 240 mg/dl high-cholesterol threshold, with a right-skewed tail. One notable outlier near 564 mg/dl (identified during Step 1's data audit) is visible as an isolated bar far from the rest of the distribution.

- **Resting Blood Pressure**: Peak occurs near the 130 mmHg Stage 1 hypertension threshold, with a meaningful portion of the cohort at or above this cutoff.

- **Max Heart Rate Achieved**: Distribution peaks around 150-160 bpm, consistent with expected values across a mixed-age cohort undergoing exercise stress testing (per the age-predicted maximum heart rate formula, 220 − age). A predicted_max_hr column was added for individualized comparison in Step 4.

## Step 4: Bivariate EDA

Compare numeric distributions between disease/no-disease groups with grouped
box plots. Compute and visualize a Pearson correlation heatmap with
**seaborn.heatmap()**.

### Question to answer:
Do people with heart disease look meaningfully differenct from people without it, on these measurements?

- A box plot condenses a distribution into a compact visual (median, quartiles, outliers) that's specifically designed for side-by-side group comparison — much easier to compare 2+ groups at a glance than overlapping histograms would be. 

- A **heatmap** gives a single visual, showing how strongly every pair of numeric variables relates to each other — including, importantly, how strongly each variable relates to condition (your disease outcome). This is often the first **_"which variables actually matter"_** signal in any clinical dataset, before building anything more sophisticated like a predictive model.

#### Seaborn: 
is a plotting library built on top of matplotlib — think of it as matplotlib's more polished, statistics-focused cousin.
It provides higher-level shortcuts specifically designed for statistical and grouped data visualization — things like comparing distributions across categories, correlation heatmaps, and grouped box plots all become one-line function calls instead of many lines of manual matplotlib setup

#### Creating boxplot

In [ ]:
# Import seaborn. 
import seaborn as sns

# Create plot grid, 2 rows and 3 columns
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Create two lists, with variables being compared and plot titles. 
variables = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
titles = ['Age', 'Resting BP', 'Cholesterol', 'Max Heart Rate', 
          'ST Depression (oldpeak)']

# For loop used to create plot in each grid

# zip() pairs up corresponding elements from the two lists, position by position,
# producing an iterable of tuples: ('age', 'Age'), ('trestbps', 'Resting BP'), etc.

# enumerate() wraps any iterable and produces (count, item) tuples,
# adding an automatic running index (0, 1, 2, ...) to whatever it's given.

for i, (var, title) in enumerate(zip(variables, titles)):
    row, col = i // 3, i % 3
    sns.boxplot(data=df, x='condition_label', y=var, ax=axes[row, col], 
                hue='condition_label', palette='Set2', legend=False)
    axes[row, col].set_title(title)
    axes[row, col].set_xlabel('')

# Hide the unused 6th subplot (we only have 5 variables for a 2x3 grid)
axes[1, 2].axis('off')

plt.tight_layout()

# Save figure plot to project subdirectory
fig.savefig('../figures/02_bivariate_boxplots.png', dpi=300, bbox_inches='tight')

# Show plot
plt.show()

#### Creating heatmnap

In [ ]:
# Create list of variables
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'condition']

# Create a matrix from result of pearson correllation
#  of list of numerical variables.
corr_matrix = df[numeric_cols].corr()

# Set figure canvas size
plt.figure(figsize=(8, 6))

# Create seaborn heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Clinical Variables')

# Save and show figure
plt.tight_layout()
plt.savefig('../figures/03_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

### What Pearson correlation measures:-
> It measures the strength and direction of a linear relationship between two **_continuous variables_**, producing a value from -1 to +1. It's the standard default **.corr()** in pandas and what you used in Step 4's heatmap.

### Why it was usable here:-
> Even though the variable "**condition**" is binary (0/1) and not continuous, Pearson correlation was used here.  When you compute a Pearson correlation between a continuous variable and a binary (0/1) variable, it's mathematically identical to something called the **_point-biserial correlation_** — a special case of Pearson correlation designed exactly for continuous-vs-binary comparisons.

> So technically, what was computed between **ca/oldpeak/thalach and condition** is **_point-biserial_** correlation, just computed via the general Pearson formula (which is what **.corr()** does automatically — it doesn't distinguish, it just runs the same linear formula regardless of whether one variable happens to be 0/1).

### Where Pearson can mislead, worth flagging as a limitation:

- **Assumes linearity** — if the true relationship between, say, thalach and disease is curved rather than a straight-line trend, Pearson would understate it
- **Sensitive to outliers** — that 564 mg/dl cholesterol outlier you flagged back in Step 1 could be nudging chol's correlation coefficient somewhat
- **ca is technically ordinal (0-3), not fully continuous** — Pearson treats it as continuous, which is a reasonable simplification but not perfectly rigorous; a purist might use Spearman's rank correlation instead, which doesn't assume linearity and handles ordinal data more properly

### Bivariate EDA Summary — Step 4

**Grouped Box Plots (disease vs. no disease):**
- **Age** and **max heart rate achieved (thalach)** both showed meaningful separation between
  groups, consistent with expectations. Thalach showed a larger visual separation than age.
- **Resting blood pressure** and **cholesterol** showed minimal separation between groups —
  a genuinely counter-intuitive finding given how heavily both are emphasized in general
  cardiac risk counseling.
- **ST depression (oldpeak)** showed strong separation, comparable to thalach, despite not
  being an initially predicted top factor.

**Correlation Heatmap (Pearson correlation with `condition`):**

| Variable | Correlation | Direction |
|---|---|---|
| ca (vessels blocked) | 0.46 | Positive |
| oldpeak (ST depression) | 0.42 | Positive |
| thalach (max heart rate) | -0.42 | Negative |
| age | 0.23 | Positive |
| trestbps (resting BP) | 0.15 | Positive |
| chol (cholesterol) | 0.08 | Positive (negligible) |

**Key clinical interpretation:**

The three strongest correlates with disease presence **(`ca`, `oldpeak`, `thalach`)** are not
traditional upstream risk factors — they are direct or near-direct measurements of disease
itself. **`ca`** reflects anatomical blockage visualized via cardiac catheterization (the gold
standard for diagnosing coronary artery disease); **`oldpeak`** reflects physiological ischemia
under exercise stress; **`thalach`** reflects the heart's functionally compromised capacity to
respond to stress. By contrast, cholesterol and blood pressure — both modifiable risk factors
routinely treated once identified — showed weak correlation with disease presence. This
likely reflects a treatment effect: cross-sectional cholesterol and blood pressure values may
represent already-managed levels rather than the untreated values that originally contributed
to disease development, diluting their apparent association with outcome in this dataset.

This distinction between *upstream, modifiable risk factors* and *downstream, direct disease
measurements* is a key theme carried forward into the clinical narrative (Step 6).

## Step 5: Age-Sex Stratified Prevalence

In [ ]:

# Bin age into decades: integer division by 10, then scale back up
# e.g. 69 -> 6 -> 60
df['age_decade'] = (df['age'] // 10 * 10).astype(int).astype(str) + 's'

# Map sex from numeric code to label for readability
df['sex_label'] = df['sex'].map({1: 'Male', 0: 'Female'})

# Group by age decade and sex, count total patients (n) and disease-present
# patients (disease) in each group
table = df.groupby(['age_decade', 'sex_label']).agg(
    n=('condition', 'size'),
    disease=('condition', 'sum')
).reset_index()

# Prevalence = disease count / total count per group, as a percentage
table['prevalence_%'] = (table['disease'] / table['n'] * 100).round(1)
table = table.sort_values(['age_decade', 'sex_label'])

print(table.to_string(index=False))
print()

# Overall prevalence across the whole cohort:
# averaging a 0/1 column gives the proportion of 1s directly
print("Overall prevalence:", round(df['condition'].mean() * 100, 1), "%")
print("Total N:", len(df))

### Age-Sex Stratified Prevalence Summary — Step 5

- **Overall disease prevalence in the cohort: 46.1%** (137 of 297 patients).
- Prevalence rises steadily with age in both sexes, from near 0% in the 20s–40s to a peak in the 60s–70s decades.
- **A consistent sex gap is present across every decade with adequate sample size:** male prevalence exceeds female prevalence by roughly 15–25 percentage points in the 40s, 50s, and 60s (e.g. 60s: 74.5% male vs. 41.2% female).
- **Caveat:** the 20s, 30s, and 70s age bins have very small sample sizes (n = 1–8 per cell). Prevalence figures in these cells are unstable and should not be interpreted as reliable estimates — they are reported for completeness, not as clinical findings.

## Step 6: Clinical Narrative


> **Background:** Identifying clinical correlates of coronary artery disease (CAD) presence can inform risk stratification prior to invasive testing. This analysis examined which clinical and diagnostic variables most strongly associate with CAD presence in a cardiac catheterization referral cohort.

> **Methods:** Given the binary outcome (CAD present/absent), Pearson correlation coefficients were computed as point-biserial correlations between disease status and available numeric clinical variables in 297 patients from the Cleveland Clinic heart disease dataset; ca, though ordinal, was treated as continuous for this purpose.

> **Results:** The three strongest correlates of disease presence were number of major vessels visualized by fluoroscopy (ca, r=0.46), ST-segment depression on exercise testing (oldpeak, r=0.42), and maximum heart rate achieved (thalach, r=-0.42). Notably, all three are direct or near-direct measurements of existing disease burden rather than upstream, modifiable risk factors. In contrast, serum cholesterol, despite elevated levels in 51% of the cohort and its prominence in general cardiac risk counseling, showed negligible correlation with disease presence (r=0.08), a counter-intuitive finding likely indicative of already-managed levels rather than untreated levels at disease onset.

> **Conclusion:** In this cohort, diagnostic markers of existing disease outperformed traditional modifiable risk factors as correlates of CAD presence, underscoring the distinction between disease detection and disease prevention in cross-sectional clinical data.